# Cross-configuration comparison: simulated dataset

Aggregates results from all 6 single-condition experiments and compares:
- Decomposition quality metrics (SIL, PNR, DR, CoV, good-unit count)
- Rate of agreement with ground truth (full recording)
- Whitening loss trajectories

**Experiments**

| # | Name | wh_mode | max_iter_b | coupling |
|---|------|---------|------------|----------|
| 01 | `kl_to_identity_iter1` | `kl_to_identity` | 1 | False |
| 02 | `kl_to_identity_iterN` | `kl_to_identity` | 5 | False |
| 03 | `kl_to_cal_iter1` | `kl_to_cal` | 1 | False |
| 04 | `kl_to_cal_iterN` | `kl_to_cal` | 5 | False |
| 05 | `kl_to_identity_iter1_coupling` | `kl_to_identity` | 1 | True |
| 06 | `kl_to_identity_iterN_coupling` | `kl_to_identity` | 5 | True |

> Run each experiment notebook first (or set `RUN_OPTIMISATION=True`) to generate the saved results this notebook loads.

In [ ]:
import sys
sys.path.insert(0, '../../..')

import json
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path

from adapt_decomp.utils import (
    rate_of_agreement_paired, rate_of_agreement,
    get_coefficient_of_variation, get_discharge_rate,
    get_pulse_to_noise_ratio, get_silhouette_measure,
    find_reliable_units,
)
from adapt_decomp.loaders import load_example

%load_ext autoreload
%autoreload 2

sns.set_theme(style='whitegrid', font_scale=1.1)


In [ ]:
RESULTS_ROOT = Path('..', '..', '..', 'data', 'JNE_data', 'sim', 'opt_results')

EXPERIMENTS = [
    dict(name='sim_01_kl_to_identity_no_coupling_iterb1', idx=1, wh_mode='kl_to_identity', sv_epochs=1, coupling=False),
    dict(name='sim_02_kl_to_identity_no_coupling_iterb5', idx=2, wh_mode='kl_to_identity', sv_epochs=5, coupling=False),
    dict(name='sim_03_kl_to_cal_no_coupling_iterb1', idx=3, wh_mode='kl_to_cal', sv_epochs=1, coupling=False),
    dict(name='sim_04_kl_to_cal_no_coupling_iterb5', idx=4, wh_mode='kl_to_cal', sv_epochs=5, coupling=False),
    dict(name='sim_05_kl_to_identity_coupling_iterb1', idx=5, wh_mode='kl_to_identity', sv_epochs=1, coupling=True),
    dict(name='sim_06_kl_to_identity_coupling_iterb5', idx=6, wh_mode='kl_to_identity', sv_epochs=5, coupling=True),
]


## 1. Load data (needed for spikes_gt and metadata)

In [ ]:
path_emg    = Path('..', '..', '..', 'data', 'JNE_data', 'sim', 'data_sim.hdf5')
path_decomp = Path('..', '..', '..', 'data', 'JNE_data', 'sim', 'decomp_sim.mat')

data     = load_example(path_emg, path_decomp, False)
n_units  = data['sep_vectors'].shape[0]
fs       = data['fs']
ext_fact = data['ext_fact']
timestamps = data['timestamps'].numpy()
print(f'Units: {n_units}  |  fs: {fs} Hz')


## 2. Load saved results

In [ ]:
def _short(name, suffix):
    """Create a human-readable condition label."""
    return f'{name}/{suffix}'

all_conds   = []  # list of (exp_name, suffix, out_dict)
roa_records = []  # per-unit RoA records

for exp in EXPERIMENTS:
    name      = exp['name']
    res_dir   = RESULTS_ROOT / name

    # ── Load outputs ─────────────────────────────────────────────────────────
    out_no = torch.load(res_dir / 'out_no.pt', weights_only=False)
    out_so = torch.load(res_dir / 'out_so.pt', weights_only=False)
    out_mo = torch.load(res_dir / 'out_mo.pt', weights_only=False)

    for suffix, out in [('no-adapt', out_no), ('single-obj', out_so), ('multi-obj', out_mo)]:
        all_conds.append((_short(name, suffix), out, name, suffix))

    # ── Load RoA summary ────────────────────────────────────────────────────
    with open(res_dir / 'roa_summary.json') as f:
        roa = json.load(f)
    for i, uid in enumerate(roa['unit_ids']):
        roa_records.append({'exp': name, 'unit': uid,
                            'roa_no': roa['roa_no'][i],
                            'roa_so': roa['roa_so'][i],
                            'roa_mo': roa['roa_mo'][i]})

df_roa = pd.DataFrame(roa_records)
print(f'Loaded {len(all_conds)} condition outputs across {len(EXPERIMENTS)} experiments.')


## 3. Quality metrics per experiment + condition

In [ ]:
rows = []
for label, out, exp_name, suffix in all_conds:
    spikes_np = out['spikes'].numpy().astype(float)
    ipts_np   = out['ipts'].numpy()

    sil  = get_silhouette_measure(spikes_np, ipts_np, ext_fact)
    pnr  = get_pulse_to_noise_ratio(spikes_np, ipts_np, ext_fact)
    dr   = get_discharge_rate(spikes_np, timestamps, discard_isi=None)
    cov  = get_coefficient_of_variation(spikes_np, timestamps, discard_isi=None)
    good = find_reliable_units(dr, cov, sil, pnr,
                               dr_low_thr=5, dr_upp_thr=35, cov_thr=0.35,
                               sil_thr=0.9, pnr_thr=30)
    for u in range(spikes_np.shape[1]):
        rows.append({'exp': exp_name, 'condition': suffix, 'label': label,
                     'unit': u, 'good': bool(good[u]),
                     'SIL': sil[u], 'PNR (dB)': pnr[u],
                     'DR (pps)': dr[u], 'CoV (%)': cov[u]})

df_metrics = pd.DataFrame(rows)
n_total    = df_metrics['unit'].nunique()

summary = (df_metrics.groupby(['exp', 'condition'])[['SIL', 'PNR (dB)', 'DR (pps)', 'CoV (%)']]
           .median().round(3))
print(summary.to_string())


In [ ]:
# Good-unit count per experiment × condition
good_counts = (df_metrics.groupby(['exp', 'condition'], sort=False)['good']
               .sum().unstack('condition')[['no-adapt', 'single-obj', 'multi-obj']])
print('\nGood units per experiment (out of', n_total, '):')
print(good_counts.to_string())


In [ ]:
# Bar chart: good units across experiments
fig, ax = plt.subplots(figsize=(14, 4.5), layout='constrained')

x        = np.arange(len(EXPERIMENTS))
w        = 0.25
suffixes = ['no-adapt', 'single-obj', 'multi-obj']
colors   = {'no-adapt': 'tab:orange', 'single-obj': 'tab:blue', 'multi-obj': 'tab:green'}

for i, suf in enumerate(suffixes):
    vals = [
        df_metrics[(df_metrics['exp'] == e['name']) & (df_metrics['condition'] == suf)]['good'].sum()
        for e in EXPERIMENTS
    ]
    bars = ax.bar(x + (i - 1) * w, vals, w, label=suf, color=colors[suf], zorder=3)

ax.axhline(n_total, color='k', linestyle='--', linewidth=1, label=f'Total (n={n_total})')
ax.set(xticks=x,
       xticklabels=[e['name'].replace('_', '\n') for e in EXPERIMENTS],
       ylabel='Good units', ylim=(0, n_total + 2),
       title='Good units per experiment  (DR ∈ [5,35] | CoV ≤ 35% | SIL ≥ 0.9 | PNR ≥ 30 dB)')
ax.legend(loc='upper right')
ax.tick_params(axis='x', labelsize=8)
plt.show()


In [ ]:
# SIL and PNR boxplots: single-obj and multi-obj across experiments
fig, axs = plt.subplots(1, 2, figsize=(14, 4), layout='constrained')

for ax, col, ylabel in [(axs[0], 'SIL', 'Silhouette measure'),
                         (axs[1], 'PNR (dB)', 'PNR (dB)')]:
    df_adapt = df_metrics[df_metrics['condition'].isin(['single-obj', 'multi-obj'])]
    sns.boxplot(data=df_adapt, x='exp', y=col, hue='condition',
                palette={'single-obj': 'tab:blue', 'multi-obj': 'tab:green'},
                width=0.5, ax=ax)
    ax.set(xlabel='', ylabel=ylabel, title=ylabel)
    ax.tick_params(axis='x', rotation=25, labelsize=8)

plt.suptitle('Quality metrics: adapted conditions across experiments', fontsize=13)
plt.show()


## 4. Rate of agreement summary

In [ ]:
# Mean RoA per experiment × condition
roa_summary = df_roa.groupby('exp').agg(
    roa_no_mean=('roa_no', lambda x: x.mean() * 100),
    roa_so_mean=('roa_so', lambda x: x.mean() * 100),
    roa_mo_mean=('roa_mo', lambda x: x.mean() * 100),
    roa_no_std =('roa_no', lambda x: x.std()  * 100),
    roa_so_std =('roa_so', lambda x: x.std()  * 100),
    roa_mo_std =('roa_mo', lambda x: x.std()  * 100),
).round(2)
print(roa_summary[['roa_no_mean', 'roa_so_mean', 'roa_mo_mean']].to_string())


In [ ]:
# Per-unit RoA: single-obj and multi-obj
fig, axs = plt.subplots(len(EXPERIMENTS), 1,
                         figsize=(12, 2.5 * len(EXPERIMENTS)),
                         layout='constrained', sharex=True)

for ax, exp in zip(axs, EXPERIMENTS):
    sub = df_roa[df_roa['exp'] == exp['name']].sort_values('unit')
    x_u = np.arange(len(sub))
    ax.plot(x_u, sub['roa_no'].values * 100, 'o-', color='tab:orange', label='no-adapt', alpha=0.7)
    ax.plot(x_u, sub['roa_so'].values * 100, 'o-', color='tab:blue',   label='single-obj', alpha=0.7)
    ax.plot(x_u, sub['roa_mo'].values * 100, 'o-', color='tab:green',  label='multi-obj', alpha=0.7)
    ax.set(ylabel='RoA (%)', ylim=(0, 105),
           title=exp['name'].replace('_', ' '))
    ax.axhline(100, color='k', linestyle=':', linewidth=0.8)
    if ax is axs[0]:
        ax.legend(loc='upper right', bbox_to_anchor=(1.18, 1), fontsize=9)

axs[-1].set(xlabel='Unit (matched)')
plt.suptitle('Rate of agreement with ground truth (full recording)', fontsize=13)
plt.show()


## 5. Whitening loss across experiments

In [ ]:
fig, axs = plt.subplots(len(EXPERIMENTS), 1,
                         figsize=(12, 2.5 * len(EXPERIMENTS)),
                         layout='constrained', sharex=True)

for ax, exp in zip(axs, EXPERIMENTS):
    name = exp['name']
    for suffix, color in [('no-adapt', 'tab:orange'),
                           ('single-obj', 'tab:blue'),
                           ('multi-obj', 'tab:green')]:
        match = [(l, o) for l, o, en, s in all_conds if en == name and s == suffix]
        if match:
            _, out = match[0]
            ax.plot(out['wh_loss'][10:-1], color=color, label=suffix, alpha=0.85)
    ax.set(ylabel='KL error²', title=name.replace('_', ' '))
    if ax is axs[0]:
        ax.legend(loc='upper right', bbox_to_anchor=(1.18, 1), fontsize=9)

axs[-1].set(xlabel='Batch')
plt.suptitle('Whitening loss across experiments', fontsize=13)
plt.show()
